<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/redes-computadores-3/02_control_congestion_tcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Control de congestión TCP: simulación AIMD interactiva

**Redes de Computadores III** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Distinguir arranque lento, evitación de congestión y recuperación rápida.
- Simular la ventana de congestión de TCP Reno.
- Explorar interactivamente el efecto de las pérdidas.
- Estimar el throughput con la fórmula de Mathis.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt

## 1. TCP Reno ronda a ronda
- **Arranque lento:** `cwnd` se duplica cada RTT hasta `ssthresh`.
- **Evitación de congestión:** +1 MSS por RTT (*additive increase*).
- **3 ACK duplicados:** `ssthresh = cwnd/2`, `cwnd = ssthresh` (*multiplicative decrease*).
- **Timeout:** `cwnd = 1`.

In [ ]:
def reno(rtts=60, ssthresh=32, perdidas=(18, 35, 48), timeouts=()):
    """cwnd (en MSS) por RTT para TCP Reno. perdidas: RTTs con 3 ACK duplicados; timeouts: RTTs con RTO."""
    cwnd, hist = 1.0, []
    for t in range(rtts):
        hist.append(cwnd)
        if t in timeouts:
            ssthresh = max(cwnd / 2, 2); cwnd = 1.0
        elif t in perdidas:
            ssthresh = max(cwnd / 2, 2); cwnd = ssthresh          # recuperación rápida
        elif cwnd < ssthresh:
            cwnd = min(cwnd * 2, ssthresh)                         # arranque lento
        else:
            cwnd += 1                                              # evitación de congestión (AIMD)
    return hist

h = reno()
plt.figure(figsize=(8, 3.5)); plt.step(range(len(h)), h, where="post")
plt.xlabel("RTT"); plt.ylabel("cwnd (MSS)"); plt.title("TCP Reno: diente de sierra AIMD"); plt.grid(alpha=.3); plt.show()

## 2. Explore
Mueva los controles: tasa de pérdidas aleatorias y `ssthresh` inicial.

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider
def explorar(p_perdida=0.03, ssthresh=32, semilla=1):
    rng = np.random.default_rng(semilla)
    perd = tuple(t for t in range(100) if rng.random() < p_perdida)
    h = reno(100, ssthresh, perd)
    plt.figure(figsize=(8, 3)); plt.step(range(100), h, where="post")
    plt.title(f"cwnd media = {np.mean(h):.1f} MSS"); plt.ylim(0, 70); plt.show()
interact(explorar, p_perdida=FloatSlider(min=0, max=0.2, step=0.01, value=0.03),
         ssthresh=IntSlider(min=4, max=64, value=32), semilla=IntSlider(min=1, max=20, value=1));

**Ejercicio 1.** Con los parámetros por defecto de `reno()`, ¿cuál es `cwnd` en el RTT número 30 (índice 30 de la lista)? → `resp_1`

In [ ]:
resp_1 = None

## 3. Fórmula de Mathis
Para pérdidas aleatorias con probabilidad $p$:
$$\text{Throughput} \approx \frac{MSS}{RTT}\cdot\frac{C}{\sqrt{p}}, \qquad C = \sqrt{3/2}$$

**Ejercicio 2.** Calcule el throughput en **Mbps** para MSS = 1460 bytes, RTT = 100 ms y $p = 0.01$. Redondee a 3 decimales → `resp_2`. ¿Cuánto cambia si el RTT baja a 10 ms (un servidor en Medellín en lugar de EE. UU.)?

In [ ]:
resp_2 = None

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 33.5)
verificar('Ejercicio 2', resp_2, 1.431, tol=1e-3)

**Ejercicio 3 (reto).** Implemente TCP **CUBIC** ($W(t) = C(t-K)^3 + W_{max}$) y compárelo con Reno en un enlace de alto producto ancho de banda × retardo.